# Dataset Preparation

**Prerequisites**
- `.env` at repo root with `XC_API_KEY=your_key`
- `uv sync` run in `building/`

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import pyrootutils
from pathlib import Path

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


# Parameters

In [3]:
N = 10
MAX_RECORDINGS = 1000
CLIPS_PER_SPECIES = 10000
MIN_XC_RECORDINGS = 100
BIRDNET_THRESHOLD = 0.92

ACTIVE_COLLECTIONS = ["diff_species"]

# Select species

In [4]:
from building.taxonomy import (
    get_species_with_recordings,
    select_same_genus,
    select_same_family,
    select_same_order,
    select_diff_order,
)

pool = get_species_with_recordings(min_recordings=MIN_XC_RECORDINGS,dl_more = True)
print(f"Species pool: {len(pool)} species with {MIN_XC_RECORDINGS} XC recordings")

Species:   0%|          | 0/11167 [00:00<?, ?it/s]

Species pool: 1805 species with 100 XC recordings


In [5]:
# show what you can choose (based on MIN_XC_RECORDINGS and your N)
from building.taxonomy import get_potential_taxa
print("Potential genera (need >=N species):")
print(get_potential_taxa("genus", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential families (need >=N distinct genera):")
print(get_potential_taxa("family", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential orders (need >=N distinct families):")
print(get_potential_taxa("order", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))


Potential genera (need >=N species):
       genus  n_species  n_distinct_genus  n_distinct_family  n_distinct_order
Phylloscopus         32                 1                  1                 1
      Turdus         29                 1                  1                 1
   Setophaga         23                 1                  1                 1
    Emberiza         17                 1                  1                 1
       Vireo         15                 1                  1                 1
      Anthus         15                 1                  1                 1
Thamnophilus         14                 1                  1                 1
     Curruca         14                 1                  1                 1
Acrocephalus         12                 1                  1                 1
    Calidris         11                 1                  1                 1
       Strix         11                 1                  1                 1
      Trogon   

In [6]:
TARGET_GENUS  = "Phylloscopus"
TARGET_FAMILY = "Tyrannidae (Tyrant Flycatchers)"
TARGET_ORDER  = "Passeriformes"

In [7]:
collections: dict[str, list[str]] = {}

if "diff_species" in ACTIVE_COLLECTIONS:
    diff_species = select_same_genus(TARGET_GENUS, N, pool)
    collections["diff_species"] = [s.scientific_name for s in diff_species]
if "diff_genus" in ACTIVE_COLLECTIONS:
    diff_genus = select_same_family(TARGET_FAMILY, N, pool)
    collections["diff_genus"] = [s.scientific_name for s in diff_genus]
if "diff_family" in ACTIVE_COLLECTIONS:
    diff_family = select_same_order(TARGET_ORDER, N, pool)
    collections["diff_family"] = [s.scientific_name for s in diff_family]
if "diff_order" in ACTIVE_COLLECTIONS:
    diff_order = select_diff_order(N, pool)
    collections["diff_order"] = [s.scientific_name for s in diff_order]

collections_to_use = {k: collections[k] for k in ACTIVE_COLLECTIONS}

for coll, names in collections.items():
    print(f"\n{coll}:")
    for n in names:
        print(f"  {n}")
print(f"\nACTIVE_COLLECTIONS (download + BirdNET): {ACTIVE_COLLECTIONS}")


diff_species:
  Phylloscopus collybita
  Phylloscopus trochilus
  Phylloscopus sibilatrix
  Phylloscopus inornatus
  Phylloscopus humei
  Phylloscopus ibericus
  Phylloscopus trochiloides
  Phylloscopus fuscatus
  Phylloscopus bonelli
  Phylloscopus xanthoschistos

ACTIVE_COLLECTIONS (download + BirdNET): ['diff_species']


# Download from Xeno-canto

In [8]:
from building.download import download_species_list

all_species = list({name for names in collections.values() for name in names})
print(f"Downloading {len(all_species)} unique species ({MAX_RECORDINGS} recordings each)…")

downloaded = download_species_list(all_species, max_recordings=MAX_RECORDINGS)

print("\nDownload summary:")
for name, paths in downloaded.items():
    print(f"  {name}: {len(paths)} files")

[Phylloscopus ibericus] existing_total=734 processed=734 existing_xc=734 target=1000


[Phylloscopus ibericus] downloaded_new=0 total_now=734 missing_after=266 skipped_duplicate=734 skipped_processed=0 skipped_quality=0 failed=0
  Phylloscopus ibericus: 734 files
[Phylloscopus bonelli] existing_total=659 processed=659 existing_xc=659 target=1000


[Phylloscopus bonelli] downloaded_new=0 total_now=659 missing_after=341 skipped_duplicate=659 skipped_processed=0 skipped_quality=0 failed=0
  Phylloscopus bonelli: 659 files
[Phylloscopus sibilatrix] existing_total=1000 processed=1000 existing_xc=1000 target=1000
[Phylloscopus sibilatrix] already complete, download=0
  Phylloscopus sibilatrix: 1000 files
[Phylloscopus trochiloides] existing_total=602 processed=602 existing_xc=602 target=1000


[Phylloscopus trochiloides] downloaded_new=0 total_now=602 missing_after=398 skipped_duplicate=602 skipped_processed=0 skipped_quality=0 failed=0
  Phylloscopus trochiloides: 602 files
[Phylloscopus humei] existing_total=661 processed=661 existing_xc=661 target=1000


[Phylloscopus humei] downloaded_new=0 total_now=661 missing_after=339 skipped_duplicate=661 skipped_processed=0 skipped_quality=0 failed=0
  Phylloscopus humei: 661 files
[Phylloscopus fuscatus] existing_total=454 processed=454 existing_xc=454 target=1000


[Phylloscopus fuscatus] downloaded_new=0 total_now=454 missing_after=546 skipped_duplicate=454 skipped_processed=0 skipped_quality=0 failed=0
  Phylloscopus fuscatus: 454 files
[Phylloscopus xanthoschistos] existing_total=727 processed=727 existing_xc=727 target=1000


[Phylloscopus xanthoschistos] downloaded_new=0 total_now=727 missing_after=273 skipped_duplicate=727 skipped_processed=0 skipped_quality=0 failed=0
  Phylloscopus xanthoschistos: 727 files
[Phylloscopus inornatus] existing_total=800 processed=800 existing_xc=800 target=1000


[Phylloscopus inornatus] downloaded_new=0 total_now=800 missing_after=200 skipped_duplicate=800 skipped_processed=0 skipped_quality=0 failed=0
  Phylloscopus inornatus: 800 files
[Phylloscopus trochilus] existing_total=1000 processed=1000 existing_xc=1000 target=1000
[Phylloscopus trochilus] already complete, download=0
  Phylloscopus trochilus: 1000 files
[Phylloscopus collybita] existing_total=1207 processed=1208 existing_xc=1207 target=1000
[Phylloscopus collybita] already complete, download=0
  Phylloscopus collybita: 1000 files

Download summary:
  Phylloscopus ibericus: 734 files
  Phylloscopus bonelli: 659 files
  Phylloscopus sibilatrix: 1000 files
  Phylloscopus trochiloides: 602 files
  Phylloscopus humei: 661 files
  Phylloscopus fuscatus: 454 files
  Phylloscopus xanthoschistos: 727 files
  Phylloscopus inornatus: 800 files
  Phylloscopus trochilus: 1000 files
  Phylloscopus collybita: 1000 files


# BirdNET validation + dataset assembly

In [9]:
from building.dataset import validate_and_build_all

dataset_paths = validate_and_build_all(
    collections_to_use,
    clips_per_species=CLIPS_PER_SPECIES,
    threshold=BIRDNET_THRESHOLD,
)

print("\nDatasets ready:")
for name, path in dataset_paths.items():
    print(f"  {name}: {path}")  


BirdNET subsamples: diff_species (10 species, threshold=0.92)...


diff_species:   0%|          | 0/10 [00:00<?, ?it/s]INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.


diff_species: 100%|██████████| 10/10 [00:00<00:00, 21.77it/s]



Building dataset: diff_species
  → /home/nathan/Documents/multi-chirp/datasets/diff_species

Datasets ready:
  diff_species: /home/nathan/Documents/multi-chirp/datasets/diff_species


# Sanity check

In [10]:
from pathlib import Path

for coll_name, root in dataset_paths.items():
    print(f"\n{coll_name}")
    for split in ("training", "validation", "testing"):
        split_dir = root / split
        if not split_dir.exists():
            continue
        species_counts = {
            d.name: len(list(d.glob("*.wav")))
            for d in sorted(split_dir.iterdir()) if d.is_dir()
        }
        total = sum(species_counts.values())
        print(f"  {split:12s} {total:5d} clips")
        for species, count in sorted(species_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"    {species:20s} {count:5d}")


diff_species
  training     42552 clips
    Phylloscopus_sibilatrix  7420
    Phylloscopus_xanthoschistos  5483
    Phylloscopus_ibericus  4981
    Phylloscopus_collybita  4960
    Phylloscopus_trochilus  4898
    Phylloscopus_bonelli  4218
    Phylloscopus_inornatus  3512
    Phylloscopus_trochiloides  2818
    Phylloscopus_humei    2385
    Phylloscopus_fuscatus  1877
  validation   10456 clips
    Phylloscopus_sibilatrix  1768
    Phylloscopus_xanthoschistos  1342
    Phylloscopus_ibericus  1224
    Phylloscopus_trochilus  1212
    Phylloscopus_collybita  1180
    Phylloscopus_bonelli  1053
    Phylloscopus_inornatus   866
    Phylloscopus_trochiloides   758
    Phylloscopus_humei     598
    Phylloscopus_fuscatus   455
  testing      10468 clips
    Phylloscopus_sibilatrix  1761
    Phylloscopus_xanthoschistos  1338
    Phylloscopus_ibericus  1220
    Phylloscopus_trochilus  1210
    Phylloscopus_collybita  1187
    Phylloscopus_bonelli  1055
    Phylloscopus_inornatus   881
    P